# Mini-CLIP: Contrastive Language-Image Pre-training (Toy Implementation)

## Paper: Radford et al. (2021) — *Learning Transferable Visual Models From Natural Language Supervision*

---

**한국어**

이 노트북은 CLIP의 핵심 아이디어를 소형 데이터로 재현합니다. 4억 개의 (이미지, 텍스트) 쌍 대신 **합성 colored shapes 데이터셋**을 사용합니다 — "red square", "blue circle" 등으로 라벨된 32×32 이미지. 작은 image encoder (CNN), 작은 text encoder (token embedding + 1-layer transformer), 그리고 공동 임베딩 공간으로의 linear projection을 정의하고, 대칭적 InfoNCE 손실로 학습합니다. 학습 후 zero-shot 분류와 image-text similarity 행렬을 시각화합니다.

**English**

This notebook reproduces CLIP's core idea on a tiny dataset. Instead of 400M (image, text) pairs we use a **synthetic colored-shapes dataset** — 32×32 images labeled "red square", "blue circle", etc. We define a small image encoder (CNN), a small text encoder (token embedding + 1-layer transformer), and linear projections to a joint embedding space, then train with the symmetric InfoNCE loss. After training, we demonstrate zero-shot classification and visualize the image–text similarity matrix.

---

## Outline / 개요

1. **Imports and reproducibility / 임포트와 재현성**
2. **Synthetic shape-color dataset / 합성 도형-색상 데이터셋**
3. **Mini text tokenizer / 미니 텍스트 토크나이저**
4. **Mini-CLIP model / Mini-CLIP 모델**
5. **Symmetric InfoNCE training / 대칭 InfoNCE 학습**
6. **Zero-shot classification / Zero-shot 분류**
7. **Similarity matrix visualization / 유사도 행렬 시각화**

## 1. Imports and reproducibility / 임포트와 재현성

**한국어** — 표준 라이브러리(numpy, matplotlib, torch)를 가져오고, 결정론적 결과를 위해 seed를 고정합니다.

**English** — Import standard libraries (numpy, matplotlib, torch) and fix random seeds for reproducibility.

In [ ]:
import math
import random
from typing import List, Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Synthetic shape-color dataset / 합성 도형-색상 데이터셋

**한국어**

학습용 데이터셋을 합성합니다. 각 이미지는 32×32 RGB 이미지에 랜덤 위치/크기로 그려진 단일 도형이며, 도형은 `circle`, `square`, `triangle` 중 하나, 색상은 `red`, `green`, `blue`, `yellow` 중 하나입니다. 캡션 형식은 CLIP 스타일의 prompt template `"a photo of a {color} {shape}"`입니다. 총 12개 (color, shape) 조합 × 다수의 위치/크기 변형으로 분포 학습이 가능합니다.

**English**

We synthesize training data: each image is a 32×32 RGB image with a single shape drawn at random position and size; shape ∈ {circle, square, triangle}, color ∈ {red, green, blue, yellow}. Captions follow the CLIP-style prompt template `"a photo of a {color} {shape}"`. With 12 (color, shape) combinations and many random placements per combination, the model can learn the joint structure.

In [ ]:
COLORS = {
    "red": (1.0, 0.0, 0.0),
    "green": (0.0, 1.0, 0.0),
    "blue": (0.0, 0.0, 1.0),
    "yellow": (1.0, 1.0, 0.0),
}
SHAPES = ["circle", "square", "triangle"]
IMAGE_SIZE = 32


def draw_shape(color_name: str, shape_name: str, rng: np.random.Generator) -> np.ndarray:
    """Render a colored shape on a black 32x32 RGB canvas.

    Args:
        color_name: Key in COLORS.
        shape_name: One of SHAPES.
        rng: NumPy random generator for placement/size.

    Returns:
        RGB image array of shape (3, 32, 32) with values in [0, 1].
    """
    img = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.float32)
    color = np.array(COLORS[color_name], dtype=np.float32)
    cx = rng.integers(10, IMAGE_SIZE - 10)
    cy = rng.integers(10, IMAGE_SIZE - 10)
    radius = rng.integers(5, 9)
    yy, xx = np.mgrid[:IMAGE_SIZE, :IMAGE_SIZE]
    if shape_name == "circle":
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 <= radius ** 2
    elif shape_name == "square":
        mask = (np.abs(xx - cx) <= radius) & (np.abs(yy - cy) <= radius)
    elif shape_name == "triangle":
        # Upward-pointing triangle: y <= cy + r and |x-cx| <= r - (cy - y)/sqrt(3)
        mask = (yy <= cy + radius) & (yy >= cy - radius) & (
            np.abs(xx - cx) <= np.maximum(0, radius - (cy + radius - yy))
        )
    else:
        raise ValueError(f"Unknown shape {shape_name}")
    img[mask] = color
    # Channel-first format expected by PyTorch.
    return img.transpose(2, 0, 1)


def make_caption(color_name: str, shape_name: str) -> str:
    """Return a CLIP-style natural-language caption."""
    return f"a photo of a {color_name} {shape_name}"


class ShapesDataset(Dataset):
    """On-the-fly synthesized colored shape images with captions."""

    def __init__(self, n_samples: int, seed: int = 0):
        self.rng = np.random.default_rng(seed)
        # Pre-sample (color, shape) labels so dataset is deterministic in length.
        self.labels = [
            (
                self.rng.choice(list(COLORS.keys())),
                self.rng.choice(SHAPES),
            )
            for _ in range(n_samples)
        ]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Tuple[np.ndarray, str]:
        color_name, shape_name = self.labels[idx]
        # New rng per sample for variability.
        local_rng = np.random.default_rng(idx + 1000)
        img = draw_shape(color_name, shape_name, local_rng)
        caption = make_caption(color_name, shape_name)
        return img, caption


# Quick visualization of a few samples.
demo = ShapesDataset(n_samples=8, seed=1)
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for ax, (img, caption) in zip(axes, demo):
    ax.imshow(img.transpose(1, 2, 0))
    ax.set_title(caption, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Mini text tokenizer / 미니 텍스트 토크나이저

**한국어**

CLIP은 BPE로 49,152 vocab을 사용하지만, 우리의 합성 캡션은 단어 종류가 적으므로 단순 word-level tokenizer로 충분합니다. 모든 캡션을 토큰화하여 vocabulary를 구축하고, `[PAD]`, `[SOS]`, `[EOS]` 특수 토큰을 추가합니다.

**English**

CLIP uses BPE with a 49,152-token vocab, but our synthetic captions have few distinct words, so a simple word-level tokenizer suffices. We tokenize all captions, build a vocabulary, and add `[PAD]`, `[SOS]`, `[EOS]` special tokens.

In [ ]:
class WordTokenizer:
    """Whitespace word-level tokenizer for our toy captions."""

    PAD = "[PAD]"
    SOS = "[SOS]"
    EOS = "[EOS]"

    def __init__(self, captions: List[str]):
        words = sorted({w for c in captions for w in c.split()})
        self.specials = [self.PAD, self.SOS, self.EOS]
        self.itos = self.specials + words
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.pad_id = self.stoi[self.PAD]
        self.sos_id = self.stoi[self.SOS]
        self.eos_id = self.stoi[self.EOS]

    @property
    def vocab_size(self) -> int:
        return len(self.itos)

    def encode(self, text: str, max_len: int = 16) -> List[int]:
        """Encode a text string to a fixed-length token id list."""
        ids = [self.sos_id] + [self.stoi[w] for w in text.split()] + [self.eos_id]
        ids = ids[:max_len]
        ids = ids + [self.pad_id] * (max_len - len(ids))
        return ids


# Build vocabulary from every possible caption template.
all_captions = [make_caption(c, s) for c in COLORS for s in SHAPES]
tokenizer = WordTokenizer(all_captions)
print("vocab:", tokenizer.itos)
print("vocab_size:", tokenizer.vocab_size)
print("example encoding:", tokenizer.encode("a photo of a red circle"))

## 4. Mini-CLIP model / Mini-CLIP 모델

**한국어**

세 가지 구성 요소:
1. **Image encoder**: 작은 CNN (Conv → ReLU → Pool 두 단계 + GAP) → linear projection.
2. **Text encoder**: token embedding + positional embedding + 1-layer Transformer encoder. `[EOS]` 위치의 출력을 representation으로 사용 — CLIP의 정확한 디자인을 따름.
3. **Joint projection**: 두 representation을 동일 차원의 임베딩 공간으로 linear projection 후 L2 정규화.

Temperature $\tau$는 학습 가능한 log-parameterized scalar.

**English**

Three components:
1. **Image encoder**: a tiny CNN (two Conv → ReLU → Pool stages + GAP) followed by a linear projection.
2. **Text encoder**: token embedding + positional embedding + 1-layer Transformer encoder. The `[EOS]` activation is used as the text representation — exactly CLIP's design.
3. **Joint projection**: linear projections of both representations to a shared embedding space, then L2 normalization.

Temperature $\tau$ is a learnable log-parameterized scalar.

In [ ]:
class TinyImageEncoder(nn.Module):
    """Two conv blocks + GAP + linear, mapping (3,32,32) images to a feature vector."""

    def __init__(self, feat_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 -> 16
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16 -> 8
            nn.Conv2d(32, feat_dim, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),  # GAP
            nn.Flatten(),  # (B, feat_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TinyTextEncoder(nn.Module):
    """Token + position embedding + 1-layer Transformer; reads off the [EOS] activation."""

    def __init__(self, vocab_size: int, max_len: int = 16, d_model: int = 64, n_heads: int = 4, eos_id: int = 2):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=0.0,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=1)
        self.layer_norm = nn.LayerNorm(d_model)
        self.eos_id = eos_id
        self.max_len = max_len

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Encode (B, L) token ids and return (B, d_model) feature at the [EOS] position."""
        b, l = token_ids.shape
        positions = torch.arange(l, device=token_ids.device).unsqueeze(0).expand(b, l)
        h = self.token_emb(token_ids) + self.pos_emb(positions)
        h = self.encoder(h)  # (B, L, d_model)
        # Pick the activation at the first [EOS] token in each row.
        eos_positions = (token_ids == self.eos_id).float().argmax(dim=1)  # (B,)
        h_eos = h[torch.arange(b), eos_positions]  # (B, d_model)
        return self.layer_norm(h_eos)


class MiniCLIP(nn.Module):
    """Mini-CLIP: image encoder + text encoder + joint projection + learnable temperature."""

    def __init__(self, vocab_size: int, embed_dim: int = 64, max_len: int = 16, eos_id: int = 2):
        super().__init__()
        self.image_encoder = TinyImageEncoder(feat_dim=embed_dim)
        self.text_encoder = TinyTextEncoder(vocab_size=vocab_size, max_len=max_len, d_model=embed_dim, eos_id=eos_id)
        self.image_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.text_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        # Temperature as log-parameterized scalar (matches CLIP, init log(1/0.07) ≈ 2.66).
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def encode_image(self, images: torch.Tensor) -> torch.Tensor:
        feats = self.image_encoder(images)
        emb = self.image_proj(feats)
        return F.normalize(emb, dim=-1)

    def encode_text(self, token_ids: torch.Tensor) -> torch.Tensor:
        feats = self.text_encoder(token_ids)
        emb = self.text_proj(feats)
        return F.normalize(emb, dim=-1)

    def forward(self, images: torch.Tensor, token_ids: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return scaled (logits_per_image, logits_per_text) similarity matrices."""
        image_emb = self.encode_image(images)
        text_emb = self.encode_text(token_ids)
        # Clamp at 100 to mirror CLIP's stability trick.
        scale = self.logit_scale.exp().clamp(max=100.0)
        logits_per_image = scale * image_emb @ text_emb.t()
        logits_per_text = logits_per_image.t()
        return logits_per_image, logits_per_text


model = MiniCLIP(vocab_size=tokenizer.vocab_size, eos_id=tokenizer.eos_id).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Mini-CLIP parameters: {n_params:,}")

## 5. Symmetric InfoNCE training / 대칭 InfoNCE 학습

**한국어**

CLIP의 핵심 손실:
$$\mathcal{L} = \frac{1}{2}\Big(\mathcal{L}_{i\to t} + \mathcal{L}_{t\to i}\Big)$$
각 방향은 대각선 인덱스가 정답인 cross-entropy. 매 배치 $N$개 쌍에서 $N$-way classification 두 개를 학습합니다.

**English**

CLIP's loss is the symmetric InfoNCE:
$$\mathcal{L} = \tfrac{1}{2}(\mathcal{L}_{i\to t} + \mathcal{L}_{t\to i})$$
Each direction is cross-entropy with diagonal indices as targets. For a batch of $N$ pairs, we run two $N$-way classifications.

In [ ]:
def collate(batch: List[Tuple[np.ndarray, str]]) -> Tuple[torch.Tensor, torch.Tensor]:
    """Batch collation: stack images and tokenize captions."""
    images, captions = zip(*batch)
    images_tensor = torch.from_numpy(np.stack(images)).float()
    token_ids = torch.tensor([tokenizer.encode(c) for c in captions], dtype=torch.long)
    return images_tensor, token_ids


def clip_loss(logits_per_image: torch.Tensor, logits_per_text: torch.Tensor) -> torch.Tensor:
    """Symmetric InfoNCE: average of image->text and text->image cross-entropy."""
    n = logits_per_image.shape[0]
    targets = torch.arange(n, device=logits_per_image.device)
    loss_i = F.cross_entropy(logits_per_image, targets)
    loss_t = F.cross_entropy(logits_per_text, targets)
    return 0.5 * (loss_i + loss_t)


train_dataset = ShapesDataset(n_samples=4096, seed=2026)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

history = []
for epoch in range(10):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for images, tokens in train_loader:
        images = images.to(DEVICE)
        tokens = tokens.to(DEVICE)
        logits_i, logits_t = model(images, tokens)
        loss = clip_loss(logits_i, logits_t)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    scheduler.step()
    avg = epoch_loss / max(1, n_batches)
    history.append(avg)
    print(f"epoch {epoch + 1:2d} | loss {avg:.4f} | logit_scale {model.logit_scale.exp().item():.2f}")

plt.figure(figsize=(5, 3))
plt.plot(history, marker="o")
plt.xlabel("epoch")
plt.ylabel("symmetric InfoNCE loss")
plt.title("Mini-CLIP training loss / 학습 손실")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Zero-shot classification / Zero-shot 분류

**한국어**

CLIP의 zero-shot 절차:
1. 가능한 모든 클래스 (12개의 (color, shape) 조합)에 대해 `"a photo of a {color} {shape}"` prompt를 생성.
2. 각 prompt를 텍스트 인코더로 임베딩 (한 번 cache).
3. 새 이미지를 image encoder로 임베딩.
4. 코사인 유사도가 가장 큰 prompt를 정답으로 선택.

테스트 데이터에서 정확도와 confusion matrix를 계산합니다.

**English**

CLIP's zero-shot procedure:
1. For every class (12 (color, shape) combinations), build the prompt `"a photo of a {color} {shape}"`.
2. Encode each prompt once with the text encoder and cache it.
3. Encode test images with the image encoder.
4. Pick the prompt with the highest cosine similarity.

We measure accuracy on a held-out test set.

In [ ]:
model.eval()

# Build the zero-shot classifier text embedding bank.
class_names = [(c, s) for c in COLORS for s in SHAPES]
class_prompts = [make_caption(c, s) for c, s in class_names]
prompt_tokens = torch.tensor([tokenizer.encode(p) for p in class_prompts], dtype=torch.long).to(DEVICE)

with torch.no_grad():
    text_bank = model.encode_text(prompt_tokens)  # (K, d_e)

print(f"Text bank shape: {text_bank.shape}")
for i, (cls, prompt) in enumerate(zip(class_names, class_prompts)):
    print(f"  class {i:2d}: {cls} -> '{prompt}'")

# Build a held-out test set with a fresh seed, ensuring different placements.
test_dataset = ShapesDataset(n_samples=512, seed=9999)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, collate_fn=collate)

true_idx_list, pred_idx_list = [], []
name_to_idx = {cls: i for i, cls in enumerate(class_names)}

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(DEVICE)
        img_emb = model.encode_image(images)  # (B, d_e)
        sims = img_emb @ text_bank.t()  # (B, K)
        preds = sims.argmax(dim=1).cpu().numpy()
        pred_idx_list.extend(preds.tolist())

for color_name, shape_name in test_dataset.labels:
    true_idx_list.append(name_to_idx[(color_name, shape_name)])

true_arr = np.array(true_idx_list)
pred_arr = np.array(pred_idx_list)
accuracy = (true_arr == pred_arr).mean()
print(f"\nZero-shot accuracy on 12-way (color × shape) classification: {accuracy:.3f}")

# Show predictions on a handful of examples.
demo_imgs, demo_caps = zip(*[test_dataset[i] for i in range(8)])
demo_imgs_t = torch.from_numpy(np.stack(demo_imgs)).float().to(DEVICE)
with torch.no_grad():
    sims = model.encode_image(demo_imgs_t) @ text_bank.t()
    demo_pred = sims.argmax(dim=1).cpu().numpy()

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for ax, img, true_caption, p_idx in zip(axes, demo_imgs, demo_caps, demo_pred):
    ax.imshow(img.transpose(1, 2, 0))
    correct = class_prompts[p_idx] == true_caption
    color = "green" if correct else "red"
    ax.set_title(f"true: {true_caption}\npred: {class_prompts[p_idx]}", fontsize=7, color=color)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Similarity matrix visualization / 유사도 행렬 시각화

**한국어**

CLIP의 학습이 어떻게 작동하는지 직관을 얻기 위해 한 미니배치의 image–text 코사인 유사도 행렬을 plot합니다. 학습이 잘 되었다면 **대각선이 가장 밝아야** 합니다 — 즉 매칭되는 (이미지, 텍스트) 쌍이 가장 높은 유사도. 또한 색상이나 도형이 같은 off-diagonal 항목은 부분적으로 밝게 나타날 수 있습니다.

**English**

To build intuition for what CLIP learns, we visualize the image-text cosine similarity matrix of a single mini-batch. After successful training, **the diagonal should be brightest** — the matched (image, text) pairs have the highest similarity. Off-diagonal entries that share a color or shape may also be partially bright.

In [ ]:
model.eval()

# Take one image of each unique class for a clean 12x12 similarity matrix.
vis_imgs, vis_caps = [], []
rng = np.random.default_rng(7)
for color_name, shape_name in class_names:
    img = draw_shape(color_name, shape_name, rng)
    vis_imgs.append(img)
    vis_caps.append(make_caption(color_name, shape_name))

vis_imgs_t = torch.from_numpy(np.stack(vis_imgs)).float().to(DEVICE)
vis_tokens = torch.tensor([tokenizer.encode(c) for c in vis_caps], dtype=torch.long).to(DEVICE)

with torch.no_grad():
    img_emb = model.encode_image(vis_imgs_t)
    text_emb = model.encode_text(vis_tokens)
    sim_matrix = (img_emb @ text_emb.t()).cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap="viridis", vmin=-1.0, vmax=1.0)
ax.set_xticks(range(len(class_names)))
ax.set_yticks(range(len(class_names)))
ax.set_xticklabels([f"{c} {s}" for c, s in class_names], rotation=45, ha="right", fontsize=8)
ax.set_yticklabels([f"{c} {s}" for c, s in class_names], fontsize=8)
ax.set_xlabel("text prompt / 텍스트 prompt")
ax.set_ylabel("image / 이미지")
ax.set_title("Mini-CLIP cosine similarity (diagonal = matched pair)\nMini-CLIP 코사인 유사도 (대각선 = 매칭 쌍)")
for i in range(sim_matrix.shape[0]):
    for j in range(sim_matrix.shape[1]):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", fontsize=6, color="white" if sim_matrix[i, j] < 0.5 else "black")
fig.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.show()

diag = np.diag(sim_matrix)
off_diag = sim_matrix[~np.eye(sim_matrix.shape[0], dtype=bool)]
print(f"Mean diagonal similarity (matched): {diag.mean():.3f}")
print(f"Mean off-diagonal similarity (mismatched): {off_diag.mean():.3f}")
print(f"Diagonal is brightest in {sum(np.argmax(sim_matrix, axis=1) == np.arange(sim_matrix.shape[0]))}/{sim_matrix.shape[0]} rows.")

## Summary / 요약

**한국어**

이 노트북은 CLIP의 핵심 메커니즘을 토이 스케일로 재현했습니다:

- **데이터**: 합성 colored shapes + 자연어 캡션 (CLIP의 (image, text) 쌍을 모방).
- **모델**: 작은 CNN image encoder + 1-layer transformer text encoder + 공동 임베딩 공간으로의 linear projection + 학습 가능한 temperature.
- **손실**: 대칭 InfoNCE — 미니배치의 $N \times N$ 유사도 행렬에서 대각선이 max가 되도록.
- **Zero-shot**: 학습 시 본 적 없을 수도 있는 도형 배치에서, 자연어 prompt와의 유사도로 분류.
- **유사도 행렬**: 학습 후 대각선이 명확히 우세 — CLIP이 의도한 정렬 학습.

실제 CLIP은 이 모델을 약 $10^7$배 확장 (400M 쌍, 367M 파라미터, 256 V100 × 12일). 그러나 알고리즘의 본질은 위의 ~15줄과 동일합니다.

**English**

This notebook reproduced CLIP's core mechanism at toy scale:

- **Data**: synthetic colored shapes + natural-language captions (mimicking CLIP's (image, text) pairs).
- **Model**: tiny CNN image encoder + 1-layer transformer text encoder + linear projection to a shared embedding + learnable temperature.
- **Loss**: symmetric InfoNCE — make the diagonal of the $N\times N$ similarity matrix the argmax of every row and column.
- **Zero-shot**: classify novel shape placements by similarity to natural-language prompts.
- **Similarity matrix**: post-training, the diagonal clearly dominates — exactly the alignment CLIP was designed to learn.

Real CLIP scales this by ~$10^7$ (400M pairs, 367M params, 256 V100 × 12 days). Yet the algorithm's essence remains the ~15 lines above.